## Cell 1 — Imports

In [19]:
# IMPORTS

import os
import random
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader,
    Subset,
    WeightedRandomSampler
)

from torchvision import transforms

import timm
from timm.utils import ModelEmaV2

from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix
)

from sklearn.model_selection import train_test_split

from tqdm import tqdm

## Cell 2 — Seed

In [20]:
# SEED

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(42)

## Cell 3 — Device

In [21]:
# DEVICE

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("🔥 DEVICE:", device)

🔥 DEVICE: cuda


## Cell 4 — Classes

In [22]:
# CLASSES

classes = [
    "anger",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

label_map = {
    c:i for i,c in enumerate(classes)
}

num_classes = len(classes)

print(label_map)

{'anger': 0, 'disgust': 1, 'fear': 2, 'happy': 3, 'neutral': 4, 'sad': 5, 'surprise': 6}


## Cell 5 — Dataset Paths

In [23]:
# DATASET PATHS

BASE = (
    "/kaggle/input/datasets/"
    "sirguthemen/"
    "preprocessed-fed-datset-collection"
)

teacher_root = os.path.join(
    BASE,
    "preprocessed_teacher_224",
    "preprocessed_teacher_224"
)

student_root = os.path.join(
    BASE,
    "preprocessed_student_96",
    "preprocessed_student_96"
)

print("Teacher root:")
print(teacher_root)

print("\nStudent root:")
print(student_root)

print("\nTeacher datasets:")
print(os.listdir(teacher_root))

print("\nStudent datasets:")
print(os.listdir(student_root))

Teacher root:
/kaggle/input/datasets/sirguthemen/preprocessed-fed-datset-collection/preprocessed_teacher_224/preprocessed_teacher_224

Student root:
/kaggle/input/datasets/sirguthemen/preprocessed-fed-datset-collection/preprocessed_student_96/preprocessed_student_96

Teacher datasets:
['Custom', 'KDEF', 'AffectNet', 'FER2013', 'CKPLUS', 'RAF-DB', 'FacialEmotionRecognition']

Student datasets:
['Custom', 'KDEF', 'AffectNet', 'FER2013', 'CKPLUS', 'RAF-DB', 'FacialEmotionRecognition']


## Cell 6 — Transforms

In [24]:
# TRANSFORMS

train_teacher_tf = transforms.Compose([

    transforms.RandomResizedCrop(
        224,
        scale=(0.75, 1.0)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.25,
        hue=0.1
    ),

    transforms.RandomRotation(15),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.25
    ),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

train_student_tf = transforms.Compose([

    transforms.RandomResizedCrop(
        128,
        scale=(0.75, 1.0)
    ),

    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.25,
        contrast=0.25,
        saturation=0.25,
        hue=0.1
    ),

    transforms.RandomRotation(15),

    transforms.ToTensor(),

    transforms.RandomErasing(
        p=0.25
    ),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_teacher_tf = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

val_student_tf = transforms.Compose([

    transforms.Resize((128,128)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

## Cell 7 — Dataset Class

In [25]:
# DATASET

class FERDataset(Dataset):

    def __init__(
        self,
        teacher_tf,
        student_tf
    ):

        self.teacher_tf = teacher_tf
        self.student_tf = student_tf

        self.samples = []

        dataset_names = os.listdir(
            teacher_root
        )

        for dataset_name in dataset_names:

            teacher_dataset = os.path.join(
                teacher_root,
                dataset_name
            )

            for emotion in classes:

                emo_folder = os.path.join(
                    teacher_dataset,
                    emotion
                )

                if not os.path.exists(
                    emo_folder
                ):
                    continue

                for fname in os.listdir(
                    emo_folder
                ):

                    if fname.lower().endswith(
                        (
                            ".jpg",
                            ".jpeg",
                            ".png"
                        )
                    ):

                        self.samples.append(
                            (
                                os.path.join(
                                    emo_folder,
                                    fname
                                ),

                                label_map[emotion]
                            )
                        )

        print(
            f"✅ TOTAL IMAGES: "
            f"{len(self.samples)}"
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        path, label = self.samples[idx]

        img = Image.open(
            path
        ).convert("RGB")

        t_img = self.teacher_tf(img)

        s_img = self.student_tf(img)

        return t_img, s_img, label

## Cell 8 — Dataset Split

In [26]:
# DATASET

full_dataset = FERDataset(
    train_teacher_tf,
    train_student_tf
)

labels = [
    s[1]
    for s in full_dataset.samples
]

indices = np.arange(len(labels))

train_idx, val_idx = train_test_split(

    indices,

    test_size=0.2,

    stratify=labels,

    random_state=42
)

train_ds = Subset(
    full_dataset,
    train_idx
)

val_full = FERDataset(
    val_teacher_tf,
    val_student_tf
)

val_ds = Subset(
    val_full,
    val_idx
)

print("Train size:", len(train_ds))
print("Validation size:", len(val_ds))

✅ TOTAL IMAGES: 42342
✅ TOTAL IMAGES: 42342
Train size: 33873
Validation size: 8469


## Cell 9 — Balanced Sampler

In [27]:
# BALANCED SAMPLER

train_labels = [
    labels[i]
    for i in train_idx
]

counts = np.bincount(
    train_labels
)

weights = 1.0 / np.sqrt(counts)

sample_weights = [
    weights[t]
    for t in train_labels
]

sampler = WeightedRandomSampler(
    sample_weights,
    len(sample_weights),
    replacement=True
)

print("Class counts:", counts)

Class counts: [4106 2296 3827 6428 6205 5820 5191]


## Cell 10 — Dataloaders

In [28]:
# DATALOADER

train_loader = DataLoader(

    train_ds,

    batch_size=64,

    sampler=sampler,

    num_workers=2,

    pin_memory=True
)

val_loader = DataLoader(

    val_ds,

    batch_size=64,

    shuffle=False,

    num_workers=2,

    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 530
Validation batches: 133


## Cell 11 — Models

In [29]:
# MODELS

teacher = timm.create_model(

    "convnext_small",

    pretrained=True,

    num_classes=num_classes

).to(device)

student = timm.create_model(

    "tf_efficientnetv2_s",

    pretrained=True,

    num_classes=num_classes

).to(device)

print("✅ Models Loaded")

✅ Models Loaded


## Cell 12 — Freeze Teacher + EMA

In [30]:
# FREEZE TEACHER

for p in teacher.parameters():

    p.requires_grad = False

teacher.eval()

# EMA

ema = ModelEmaV2(
    student,
    decay=0.999
)

print("✅ Teacher Frozen")
print("✅ EMA Ready")

✅ Teacher Frozen
✅ EMA Ready


## Cell 13 — Feature Projector

In [31]:
# FEATURE PROJECTOR

class FeatureProjector(nn.Module):

    def __init__(
        self,
        in_ch,
        out_ch
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_ch,
                512,
                kernel_size=1
            ),

            nn.BatchNorm2d(512),

            nn.GELU(),

            nn.Conv2d(
                512,
                out_ch,
                kernel_size=1
            )
        )

    def forward(self, x):

        return self.block(x)

## Cell 14 — Feature Dimensions

In [32]:
# FEATURE DIMENSIONS

with torch.no_grad():

    dummy_t = torch.randn(
        1,3,224,224
    ).to(device)

    dummy_s = torch.randn(
        1,3,128,128
    ).to(device)

    t_feat = teacher.forward_features(
        dummy_t
    )

    s_feat = student.forward_features(
        dummy_s
    )

t_dim = t_feat.shape[1]
s_dim = s_feat.shape[1]

print("Teacher dim:", t_dim)
print("Student dim:", s_dim)

projector = FeatureProjector(
    s_dim,
    t_dim
).to(device)

Teacher dim: 768
Student dim: 1280


## Cell 15 — Focal Loss

In [33]:
# FOCAL LOSS

class FocalLoss(nn.Module):

    def __init__(
        self,
        gamma=2
    ):

        super().__init__()

        self.gamma = gamma

    def forward(
        self,
        logits,
        targets
    ):

        ce = F.cross_entropy(
            logits,
            targets,
            reduction='none',
            label_smoothing=0.1
        )

        pt = torch.exp(-ce)

        loss = (
            (1 - pt) ** self.gamma
        ) * ce

        return loss.mean()

## Cell 16 — KD Loss

In [34]:
# KD LOSS

class KDLoss(nn.Module):

    def __init__(
        self,
        T=4.0,
        alpha=0.5
    ):

        super().__init__()

        self.T = T
        self.alpha = alpha

        self.ce = FocalLoss()

    def forward(

        self,

        s_logits,
        t_logits,

        labels,

        s_feat,
        t_feat
    ):

        ce_loss = self.ce(
            s_logits,
            labels
        )

        kd_loss = F.kl_div(

            F.log_softmax(
                s_logits / self.T,
                dim=1
            ),

            F.softmax(
                t_logits / self.T,
                dim=1
            ),

            reduction='batchmean'

        ) * (self.T ** 2)

        s_feat = projector(s_feat)

        t_feat = F.adaptive_avg_pool2d(
            t_feat,
            s_feat.shape[-2:]
        )

        feat_loss = (
            1 -
            F.cosine_similarity(
                s_feat.flatten(1),
                t_feat.flatten(1)
            ).mean()
        )

        total = (

            0.4 * ce_loss +

            0.4 * kd_loss +

            0.2 * feat_loss
        )

        return total

loss_fn = KDLoss()

print("✅ KD Loss Ready")

✅ KD Loss Ready


## Cell 17 — Optimizer

In [35]:
# =========================================================
# OPTIMIZER
# =========================================================

optimizer = optim.AdamW(

    list(student.parameters()) +
    list(projector.parameters()),

    lr=2e-4,

    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=30
)

scaler = torch.amp.GradScaler("cuda")

print("✅ Optimizer Ready")

✅ Optimizer Ready


In [36]:
print(train_loader)

## Cell 18 — Training

In [37]:
# TRAINING

best_acc = 0

epochs = 30

print("\n🚀 TRAINING STARTED\n")

for epoch in range(epochs):

    student.train()
    projector.train()

    pbar = tqdm(
        train_loader
    )

    running_loss = 0

    for t_img, s_img, y in pbar:

        t_img = t_img.to(device)
        s_img = s_img.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            with torch.no_grad():

                t_out = teacher(
                    t_img
                )

                t_feat = teacher.forward_features(
                    t_img
                )

            s_out = student(
                s_img
            )

            s_feat = student.forward_features(
                s_img
            )

            loss = loss_fn(

                s_out,
                t_out,

                y,

                s_feat,
                t_feat
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            student.parameters(),
            1.0
        )

        scaler.step(optimizer)

        scaler.update()

        ema.update(student)

        running_loss += loss.item()

        pbar.set_description(

            f"Epoch {epoch+1} "
            f"Loss {loss.item():.4f}"
        )

    scheduler.step()

    # VALIDATION

    ema.module.eval()

    preds = []
    gts = []

    with torch.no_grad():

        for _, s_img, y in val_loader:

            s_img = s_img.to(device)
            y = y.to(device)

            with torch.amp.autocast("cuda"):

                out1 = ema.module(
                    s_img
                )

                out2 = ema.module(
                    torch.flip(
                        s_img,
                        dims=[3]
                    )
                )

                out = (
                    out1 + out2
                ) / 2

            pred = out.argmax(1)

            preds.extend(
                pred.cpu().numpy()
            )

            gts.extend(
                y.cpu().numpy()
            )

    acc = (
        balanced_accuracy_score(
            gts,
            preds
        ) * 100
    )

    print(
        f"\n📊 Epoch {epoch+1}"
    )

    print(
        f"Balanced Accuracy:"
        f" {acc:.2f}%"
    )

    print(
        confusion_matrix(
            gts,
            preds
        )
    )

    # SAVE

    if acc > best_acc:

        best_acc = acc

        torch.save(

            ema.module.state_dict(),

            "best_fer_student.pth"
        )

        print(
            "💾 BEST MODEL SAVED"
        )

print("\n🏁 TRAINING FINISHED")

print(
    f"\n🏆 BEST ACCURACY:"
    f" {best_acc:.2f}%"
)


🚀 TRAINING STARTED



Epoch 1 Loss 0.5097: 100%|██████████| 530/530 [06:34<00:00,  1.34it/s] 



📊 Epoch 1
Balanced Accuracy: 20.53%
[[510 256   1 103  69  61  27]
 [182 201   0  50  49  68  24]
 [461 228   0 115  63  36  54]
 [330 412   0 350 312 121  82]
 [489 457   0 151 283 116  55]
 [552 431   1 125 145 160  41]
 [647 248   0 125  90  84 104]]
💾 BEST MODEL SAVED


Epoch 2 Loss 0.4695: 100%|██████████| 530/530 [05:41<00:00,  1.55it/s]



📊 Epoch 2
Balanced Accuracy: 44.02%
[[ 517  166    9   33  199   91   12]
 [ 128  239    0   17  115   68    7]
 [ 338   99   61   54  187  125   93]
 [  68   89    2  954  394   78   22]
 [ 160   90    0   23 1197   71   10]
 [ 223  188    9   24  523  470   18]
 [ 300   81    9   54  255   68  531]]
💾 BEST MODEL SAVED


...


📊 Epoch 28
Balanced Accuracy: 70.28%
[[ 679   51   66   16   46  127   42]
 [  88  349   16    6   30   62   23]
 [ 105   13  475   11   41  150  162]
 [  13    8   10 1403   93   30   50]
 [  45   22   19   54 1245  139   27]
 [ 126   43   86   12  140  994   54]
 [  40   20   87   20   41   58 1032]]


Epoch 29 Loss 0.3638: 100%|██████████| 530/530 [05:44<00:00,  1.54it/s]



📊 Epoch 29
Balanced Accuracy: 70.15%
[[ 677   52   65   17   47  126   43]
 [  89  350   16    6   29   61   23]
 [ 105   13  472   11   41  152  163]
 [  15    8   10 1398   91   32   53]
 [  45   19   19   54 1245  142   27]
 [ 123   45   87   12  142  993   53]
 [  42   21   88   20   40   57 1030]]


Epoch 30 Loss 0.2749: 100%|██████████| 530/530 [05:43<00:00,  1.54it/s]



📊 Epoch 30
Balanced Accuracy: 70.08%
[[ 673   53   64   15   49  128   45]
 [  89  348   17    7   29   61   23]
 [ 105   13  470   11   41  155  162]
 [  15    7   10 1396   91   33   55]
 [  46   18   18   54 1247  141   27]
 [ 124   44   83   12  140  997   55]
 [  42   21   87   20   40   56 1032]]

🏁 TRAINING FINISHED

🏆 BEST ACCURACY: 72.13%


## Cell 19 — Load Best Model

In [38]:
# LOAD BEST

student.load_state_dict(

    torch.load(
        "best_fer_student.pth",
        map_location=device
    )
)

student.eval()

print("✅ Best model loaded")

✅ Best model loaded


In [40]:
!pip install onnx onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 11.3 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 11.0 MB/s eta 0:00:00


## Cell 20 — Export ONNX

In [48]:
# EXPORT ONNX 

import warnings
import os
import torch

#  Hide warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

dummy = torch.randn(
    1,
    3,
    128,
    128
).to(device)

torch.onnx.export(

    student,
    dummy,
    "best_fer_student.onnx",

    export_params=True,

    opset_version=18,  
    do_constant_folding=True,

    input_names=["input"],

    output_names=["output"]
)

print("\n✅ ONNX EXPORTED")

W0529 14:04:01.486000 133 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0529 14:04:01.487000 133 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0529 14:04:01.489000 133 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0529 14:04:01.491000 133 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅

✅ ONNX EXPORTED


In [49]:
import os

print(os.listdir("/kaggle/working"))

['best_fer_student.onnx.data', 'best_fer_student.pth', '.virtual_documents', 'best_fer_student.onnx']
